# Protein Baseline Features (UniProt + Pfam)

This notebook builds a **traditional baseline feature set** for protein family classification
using the processed UniProt+Pfam dataset.

**Input:** `data/processed/protein_uniprot_pfam_top{N_FAM}_per{PER_FAM}.csv`  
Columns: `accession, sequence, length, family`

**Output:**
- Feature table (CSV): `data/processed/protein_features_baseline_top{N_FAM}_per{PER_FAM}.csv`
- Summary (JSON): `reports/protein_baseline_features_summary.json`

## Feature families included

### 1) Basic + Composition
- length
- 20 amino-acid fractions
- grouped composition fractions (hydrophobic/polar/charged/aromatic/etc.)
- Shannon entropy of AA distribution
- max homopolymer run

### 2) Physicochemical (BioPython ProtParam)
- molecular weight
- aromaticity
- instability index
- isoelectric point (pI)
- GRAVY hydropathy
- estimated secondary structure fractions (helix/turn/sheet)
- charge at pH 7.0

### 3) Sequence-order / “traditional bioinformatics” descriptors
- CTD (Composition–Transition–Distribution) across:
  - hydrophobicity (3-class)
  - polarity (2-class)
  - charge (3-class)
- PseAAC (Type I) using 3 properties (hydrophobicity, hydrophilicity, mass)
- Reduced alphabet dipeptides (7 groups → 49 2-mer frequency features)

Notes:
- Secondary structure features here are **propensity-based estimates** (BioPython),
  not experimental structures.

In [8]:
from pathlib import Path
import json
import random
import sys
import platform
from collections import Counter
from itertools import product

import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm

from Bio.SeqUtils.ProtParam import ProteinAnalysis

/opt/anaconda3/envs/bioseq-capstone/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths + config + load dataset

This notebook assumes it is stored under `notebooks/`, so:

`ROOT = Path.cwd().parents[0]`

It reads parameters from `configs/config.yaml`.

In [9]:
ROOT = Path.cwd().parents[0]
DATA = ROOT / "data"
PROCESSED = DATA / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
CONFIGS = ROOT / "configs"

for p in [PROCESSED, REPORTS, FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

cfg_path = CONFIGS / "config.yaml"
assert cfg_path.exists(), f"Missing config: {cfg_path}"

with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

SEED = int(cfg["project"]["random_seed"])
N_FAM = int(cfg["protein"]["n_families"])
PER_FAM = int(cfg["protein"]["per_family"])
MAX_LEN = int(cfg["protein"]["max_len_aa"])

random.seed(SEED)
np.random.seed(SEED)

in_csv = PROCESSED / f"protein_uniprot_pfam_top{N_FAM}_per{PER_FAM}.csv"
assert in_csv.exists(), f"Missing processed dataset: {in_csv}"

df = pd.read_csv(in_csv)
df.shape, df.head(3)

((2293, 4),
   accession                                           sequence  length  \
 0    Q96RD1  MRNHTEITEFILLGLTDDPNFQVVIFVFLLITYMLSITGNLTLITI...     312   
 1    Q9H210  MRQINQTQVTEFLLLGLSDGPHTEQLLFIVLLGVYLVTVLGNLLLI...     308   
 2    Q8NGZ3  MNHSVVTEFIILGLTKKPELQGIIFLFFLIVYLVAFLGNMLIIIAK...     307   
 
     family  
 0  PF13853  
 1  PF13853  
 2  PF13853  )

## Basic validation checks

We verify:
- required columns exist
- no missing values
- lengths match sequence string lengths
- sequences contain only 20 canonical amino acids

In [10]:
required = ["accession", "sequence", "length", "family"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

na_counts = df[required].isna().sum()
print("NA counts:\n", na_counts)

# length consistency
seq_len = df["sequence"].astype(str).str.len()
n_mismatch = int((seq_len != df["length"]).sum())
print("Length mismatches:", n_mismatch)
assert n_mismatch == 0, "Found sequence length mismatches vs 'length' column"

# canonical AA check
AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")
AA_SET = set(AA_LIST)

def count_noncanonical(seq: str) -> int:
    return sum(1 for ch in seq if ch not in AA_SET)

df["noncanonical_count"] = df["sequence"].astype(str).apply(count_noncanonical)
n_bad = int((df["noncanonical_count"] > 0).sum())
print("Sequences with noncanonical chars:", n_bad)
assert n_bad == 0, "Found non-canonical amino acids. Consider filtering or cleaning."

# duplicates info (not necessarily an error, but good to report)
dup_accessions = int(df["accession"].duplicated().sum())
dup_seq_family = int(df.duplicated(subset=["sequence", "family"]).sum())
print("Duplicates:", {"dup_accessions": dup_accessions, "dup_sequence_family": dup_seq_family})

df = df.drop(columns=["noncanonical_count"])
df.shape

NA counts:
 accession    0
sequence     0
length       0
family       0
dtype: int64
Length mismatches: 0
Sequences with noncanonical chars: 0
Duplicates: {'dup_accessions': 0, 'dup_sequence_family': 0}


(2293, 4)

## Feature helper functions

We define:
- Shannon entropy of AA distribution
- max homopolymer run length
- grouped composition fractions
- reduced alphabet mapping for dipeptides (7-group alphabet)

In [11]:
def shannon_entropy_from_counts(counts: dict, total: int) -> float:
    if total <= 0:
        return 0.0
    ent = 0.0
    for v in counts.values():
        if v <= 0:
            continue
        p = v / total
        ent -= p * np.log2(p)
    return float(ent)

def max_run_length(seq: str) -> int:
    if not seq:
        return 0
    best = 1
    cur = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i-1]:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return int(best)

# grouped composition (classic)
AA_GROUPS = {
    "hydrophobic": set("AILMFWVY"),     # includes aromatic
    "polar": set("STNQCY"),
    "positive": set("KRH"),
    "negative": set("DE"),
    "aromatic": set("FWY"),
    "aliphatic": set("AILV"),
    "small": set("AGSTP"),
    "sulfur": set("CM"),
    "amide": set("NQ"),
}

def grouped_fractions(seq: str) -> dict:
    L = len(seq)
    counts = Counter(seq)
    out = {}
    for g, aset in AA_GROUPS.items():
        out[f"frac_group_{g}"] = float(sum(counts[a] for a in aset) / L)
    return out

# reduced alphabet (7 groups) for dipeptides
# This is a common baseline trick to keep k-mer features small.
RED7 = {
    "A": "A", "G": "A", "V": "A",
    "I": "B", "L": "B", "F": "B", "P": "B",
    "Y": "C", "M": "C", "T": "C", "S": "C",
    "H": "D", "N": "D", "Q": "D", "W": "D",
    "R": "E", "K": "E",
    "D": "F", "E": "F",
    "C": "G",
}
RED7_SYMBOLS = sorted(set(RED7.values()))  # 7 symbols

def to_red7(seq: str) -> str:
    return "".join(RED7.get(ch, "X") for ch in seq)

def red7_dipeptide_freq(seq: str) -> dict:
    rs = to_red7(seq)
    L = len(rs)
    out = {}
    # total dipeptides
    denom = max(L - 1, 1)
    counts = Counter(rs[i:i+2] for i in range(L - 1))
    for a, b in product(RED7_SYMBOLS, RED7_SYMBOLS):
        key = f"red7_di_{a}{b}"
        out[key] = float(counts.get(a + b, 0) / denom)
    return out

## CTD (Composition–Transition–Distribution)

We compute CTD descriptors across three classic partitions:
- hydrophobicity (3 groups)
- polarity (2 groups)
- charge (3 groups)

CTD includes:
- Composition: fraction of residues in each group
- Transition: rate of adjacent group changes
- Distribution: positions (% length) for 1st / 25% / 50% / 75% / 100% occurrence of each group

In [12]:
CTD_GROUPS = {
    "hydrophobicity_3": {
        "H": set("AILMFWV"),
        "P": set("CNQSTY"),
        "N": set("DEGHRKP"),
    },
    "polarity_2": {
        "P": set("STNQCYW"),
        "N": set("ADFGHIKLMPVR"),
    },
    "charge_3": {
        "+": set("KRH"),
        "-": set("DE"),
        "0": set("ACFGILMNPQSTVWY"),
    }
}

def _assign_group(seq: str, group_def: dict) -> str:
    inv = {}
    for g, aset in group_def.items():
        for a in aset:
            inv[a] = g
    return "".join(inv.get(ch, "X") for ch in seq)

def ctd_composition(group_seq: str, symbols: list[str]) -> dict:
    L = len(group_seq)
    c = Counter(group_seq)
    return {f"ctd_comp_{s}": float(c.get(s, 0) / L) for s in symbols}

def ctd_transition(group_seq: str, symbols: list[str]) -> dict:
    L = len(group_seq)
    out = {f"ctd_trans_{a}{b}": 0.0 for i, a in enumerate(symbols) for b in symbols[i+1:]}
    if L < 2:
        return out

    trans = Counter()
    for i in range(L - 1):
        a, b = group_seq[i], group_seq[i+1]
        if a == b:
            continue
        key = "".join(sorted([a, b]))
        trans[key] += 1

    denom = (L - 1)
    for i, a in enumerate(symbols):
        for b in symbols[i+1:]:
            key = "".join(sorted([a, b]))
            out[f"ctd_trans_{a}{b}"] = float(trans.get(key, 0) / denom)
    return out

def ctd_distribution(group_seq: str, symbols: list[str]) -> dict:
    L = len(group_seq)
    out = {}
    for s in symbols:
        idxs = [i for i, ch in enumerate(group_seq, start=1) if ch == s]  # 1-indexed
        if not idxs:
            for q in [1, 25, 50, 75, 100]:
                out[f"ctd_dist_{s}_{q}"] = 0.0
            continue

        n = len(idxs)
        picks = [
            idxs[0],
            idxs[int(np.ceil(0.25 * n)) - 1],
            idxs[int(np.ceil(0.50 * n)) - 1],
            idxs[int(np.ceil(0.75 * n)) - 1],
            idxs[-1],
        ]
        for q, pos in zip([1, 25, 50, 75, 100], picks):
            out[f"ctd_dist_{s}_{q}"] = float(100.0 * pos / L)
    return out

def ctd_features(seq: str) -> dict:
    feats = {}
    for name, gdef in CTD_GROUPS.items():
        gs = _assign_group(seq, gdef)
        symbols = list(gdef.keys())
        feats.update({f"{name}__{k}": v for k, v in ctd_composition(gs, symbols).items()})
        feats.update({f"{name}__{k}": v for k, v in ctd_transition(gs, symbols).items()})
        feats.update({f"{name}__{k}": v for k, v in ctd_distribution(gs, symbols).items()})
    return feats

## PseAAC (Type I)

PseAAC adds **sequence-order information** via autocorrelation of amino-acid properties.

We use 3 common property scales:
- Hydrophobicity (Kyte–Doolittle)
- Hydrophilicity (Hopp–Woods)
- Residue mass

Parameters:
- **λ (lambda):** number of correlation lags
- **weight:** how much sequence-order terms contribute

This returns:
- 20 normalized composition terms
- λ correlation terms

In [13]:
PSEAAC_PROPS = {
    "hydrophobicity": {  # Kyte-Doolittle
        "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
        "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
        "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
        "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
    },
    "hydrophilicity": {  # Hopp-Woods
        "A": -0.5, "C": -1.0, "D": 3.0, "E": 3.0, "F": -2.5,
        "G": 0.0, "H": -0.5, "I": -1.8, "K": 3.0, "L": -1.8,
        "M": -1.3, "N": 0.2, "P": 0.0, "Q": 0.2, "R": 3.0,
        "S": 0.3, "T": -0.4, "V": -1.5, "W": -3.4, "Y": -2.3,
    },
    "mass": {  # residue masses approx
        "A": 89.09, "C": 121.15, "D": 133.10, "E": 147.13, "F": 165.19,
        "G": 75.07, "H": 155.16, "I": 131.17, "K": 146.19, "L": 131.17,
        "M": 149.21, "N": 132.12, "P": 115.13, "Q": 146.15, "R": 174.20,
        "S": 105.09, "T": 119.12, "V": 117.15, "W": 204.23, "Y": 181.19,
    }
}

def _zscore_prop(prop: dict) -> dict:
    vals = np.array([prop[a] for a in AA_LIST], dtype=float)
    mu, sd = vals.mean(), vals.std()
    return {a: float((prop[a] - mu) / (sd + 1e-12)) for a in AA_LIST}

PSEAAC_PROPS_Z = {name: _zscore_prop(prop) for name, prop in PSEAAC_PROPS.items()}
PSEAAC_PROP_LIST = list(PSEAAC_PROPS_Z.values())

def pse_aac(seq: str, lam: int = 10, weight: float = 0.05) -> dict:
    L = len(seq)
    c = Counter(seq)
    f = np.array([c[a] / L for a in AA_LIST], dtype=float)

    lam = min(lam, L - 1) if L > 1 else 0
    thetas = []
    for k in range(1, lam + 1):
        vals = []
        for prop in PSEAAC_PROP_LIST:
            diffs = [(prop[seq[i]] - prop[seq[i + k]]) ** 2 for i in range(L - k)]
            vals.append(float(np.mean(diffs)) if diffs else 0.0)
        thetas.append(float(np.mean(vals)) if vals else 0.0)

    denom = 1.0 + weight * (sum(thetas) if thetas else 0.0)

    out = {}
    for i, a in enumerate(AA_LIST):
        out[f"pse_aac_{a}"] = float(f[i] / denom)

    for k, th in enumerate(thetas, start=1):
        out[f"pse_theta_{k}"] = float((weight * th) / denom)

    return out

## Feature extraction function (single sequence)

We combine:
- composition (AA fractions + grouped)
- ProtParam physicochemical features
- estimated secondary-structure fractions (BioPython)
- entropy + max run length
- CTD + PseAAC
- reduced alphabet dipeptides (49 features)

In [14]:
ADD_CTD = True
ADD_PSEAAC = True
PSEAAC_LAMBDA = 10
PSEAAC_WEIGHT = 0.05
ADD_RED7_DIPEPTIDES = True

def protein_features(seq: str) -> dict:
    seq = str(seq).strip().upper()
    L = len(seq)

    feats = {"seq_len": int(L)}

    # 20 AA fractions
    c = Counter(seq)
    for a in AA_LIST:
        feats[f"frac_{a}"] = float(c[a] / L)

    # grouped fractions
    feats.update(grouped_fractions(seq))

    # entropy + low complexity
    feats["aa_entropy"] = shannon_entropy_from_counts({a: c[a] for a in AA_LIST}, L)
    feats["max_homopolymer_run"] = max_run_length(seq)
    feats["n_unique_aas"] = int(len({ch for ch in seq}))

    # ProtParam features
    pa = ProteinAnalysis(seq)
    feats["molecular_weight"] = float(pa.molecular_weight())
    feats["aromaticity"] = float(pa.aromaticity())
    feats["instability_index"] = float(pa.instability_index())
    feats["isoelectric_point"] = float(pa.isoelectric_point())
    feats["gravy"] = float(pa.gravy())
    feats["charge_ph7"] = float(pa.charge_at_pH(7.0))

    # Secondary structure (propensity estimate)
    helix, turn, sheet = pa.secondary_structure_fraction()
    feats["secstruct_helix"] = float(helix)
    feats["secstruct_turn"] = float(turn)
    feats["secstruct_sheet"] = float(sheet)

    # CTD
    if ADD_CTD:
        feats.update(ctd_features(seq))

    # PseAAC
    if ADD_PSEAAC:
        feats.update(pse_aac(seq, lam=PSEAAC_LAMBDA, weight=PSEAAC_WEIGHT))

    # Reduced alphabet dipeptides
    if ADD_RED7_DIPEPTIDES:
        feats.update(red7_dipeptide_freq(seq))

    return feats

## Build the feature table

We compute features for every row and save:

- `data/processed/protein_features_baseline_top{N_FAM}_per{PER_FAM}.csv`

The saved CSV includes:
- identifiers: accession, family
- numeric feature columns

In [15]:
tqdm.pandas()

rows = []
for _, r in tqdm(df.iterrows(), total=len(df), desc="Extracting protein features"):
    feats = protein_features(r["sequence"])
    feats["accession"] = r["accession"]
    feats["family"] = r["family"]
    rows.append(feats)

feat_df = pd.DataFrame(rows)

# move id columns first
id_cols = ["accession", "family"]
other_cols = [c for c in feat_df.columns if c not in id_cols]
feat_df = feat_df[id_cols + other_cols]

feat_df.shape, feat_df.head(2)

Extracting protein features: 100%|█████████| 2293/2293 [00:06<00:00, 356.92it/s]


((2293, 178),
   accession   family  seq_len    frac_A    frac_C    frac_D    frac_E  \
 0    Q96RD1  PF13853      312  0.044872  0.032051  0.028846  0.016026   
 1    Q9H210  PF13853      308  0.068182  0.029221  0.022727  0.016234   
 
      frac_F    frac_G    frac_H  ...  red7_di_FE  red7_di_FF  red7_di_FG  \
 0  0.096154  0.025641  0.019231  ...    0.009646    0.003215         0.0   
 1  0.058442  0.042208  0.022727  ...    0.006515    0.000000         0.0   
 
    red7_di_GA  red7_di_GB  red7_di_GC  red7_di_GD  red7_di_GE  red7_di_GF  \
 0    0.003215    0.009646    0.009646    0.000000    0.006431    0.003215   
 1    0.013029    0.003257    0.003257    0.006515    0.000000    0.003257   
 
    red7_di_GG  
 0         0.0  
 1         0.0  
 
 [2 rows x 178 columns])

## Sanity checks on the feature table

We verify:
- no missing values
- all feature columns are numeric (except accession/family)
- no infinities

In [16]:
assert feat_df["accession"].isna().sum() == 0
assert feat_df["family"].isna().sum() == 0

feature_cols = [c for c in feat_df.columns if c not in ["accession", "family"]]

na_feat = feat_df[feature_cols].isna().sum().sum()
print("Total NA in features:", int(na_feat))
assert na_feat == 0, "Found missing values in feature columns."

# numeric check
non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(feat_df[c])]
print("Non-numeric feature cols:", non_numeric)
assert len(non_numeric) == 0, f"Non-numeric feature columns found: {non_numeric}"

# finite check
finite_ok = np.isfinite(feat_df[feature_cols].to_numpy()).all()
print("All finite:", finite_ok)
assert finite_ok, "Found inf or -inf in feature matrix."

feat_df.shape

Total NA in features: 0
Non-numeric feature cols: []
All finite: True


(2293, 178)

## Save processed features + summary JSON

In [17]:
out_csv = PROCESSED / f"protein_features_baseline_top{N_FAM}_per{PER_FAM}.csv"
feat_df.to_csv(out_csv, index=False)

summary = {
    "input_dataset": str(in_csv),
    "output_features_csv": str(out_csv),
    "n_rows": int(feat_df.shape[0]),
    "n_families": int(feat_df["family"].nunique()),
    "families": feat_df["family"].value_counts().index.tolist(),
    "family_counts": feat_df["family"].value_counts().to_dict(),
    "n_features": int(len(feature_cols)),
    "feature_groups": {
        "aa_fractions_20": True,
        "grouped_fractions": True,
        "protparam_physicochem": True,
        "secondary_structure_fraction": True,
        "entropy_and_low_complexity": True,
        "ctd_enabled": bool(ADD_CTD),
        "pse_aac_enabled": bool(ADD_PSEAAC),
        "pse_aac_lambda": int(PSEAAC_LAMBDA) if ADD_PSEAAC else None,
        "pse_aac_weight": float(PSEAAC_WEIGHT) if ADD_PSEAAC else None,
        "red7_dipeptides_enabled": bool(ADD_RED7_DIPEPTIDES),
    },
    "config": {
        "seed": int(SEED),
        "n_families_top": int(N_FAM),
        "per_family_target": int(PER_FAM),
        "max_len_aa": int(MAX_LEN),
    },
    "validation": {
        "missing_values_features": int(na_feat),
        "finite_features": bool(finite_ok),
        "dup_accessions_in_input": int(dup_accessions),
        "dup_sequence_family_in_input": int(dup_seq_family),
    },
    "env": {
        "python_version": sys.version.split()[0],
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}

summary_path = REPORTS / "protein_baseline_features_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

out_csv, summary_path

(PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone/data/processed/protein_features_baseline_top10_per400.csv'),
 PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone/reports/protein_baseline_features_summary.json'))

## Quick peek: top-level stats

This is just a quick check to confirm expected feature ranges.

In [18]:
feat_df[["seq_len", "molecular_weight", "isoelectric_point", "gravy", "instability_index",
         "secstruct_helix", "secstruct_sheet", "aa_entropy", "max_homopolymer_run"]].describe()

,seq_len,molecular_weight,isoelectric_point,gravy,instability_index,secstruct_helix,secstruct_sheet,aa_entropy,max_homopolymer_run
count,2293.000000,2293.000000,2293.000000,2293.000000,2293.000000,2293.000000,2293.000000,2293.000000,2293.000000
mean,394.406891,44219.679177,8.146660,-0.199004,46.791249,0.304038,0.358842,4.061796,3.642826
std,196.080158,22190.089971,1.460523,0.615275,12.161092,0.037972,0.091255,0.096399,2.052802
min,73.000000,8260.048900,4.050028,-2.046763,14.198305,0.082770,0.133603,3.397664,2.000000
25%,292.000000,32141.993700,6.971211,-0.711173,38.942120,0.281150,0.284810,4.023736,3.000000
50%,346.000000,38545.106700,8.700540,-0.311017,45.207783,0.302714,0.345411,4.077203,3.000000
75%,500.000000,56413.641700,9.173416,0.375275,52.594457,0.327434,0.438776,4.128292,4.000000
max,1023.000000,118101.326700,11.860652,1.015436,137.833613,0.451411,0.569079,4.258716,21.000000
